In [61]:
import spacy
import pandas as pd
import re
from tqdm import tqdm
nlp = spacy.load("en_core_web_sm")

In [55]:
df = pd.read_csv("processed_v2.csv")

# збираємо речення
sentences = df.groupby("sent_id")["token"].apply(list)
sentences = sentences.apply(lambda x: " ".join(x)).tolist()

texts = sentences

print("TOTAL SENTENCES:", len(texts))
print("SAMPLE:")
print(texts[0])

TOTAL SENTENCES: 7142
SAMPLE:
Шановні колеги , Рада національної безпеки має Секретаря Ради національної безпеки , найближчим часом будуть сформовані фактично всі підрозділи РНБО , призначені заступники Секретаря Ради національної безпеки та оборони , і я сподіваюся , що Рада національної безпеки та оборони буде відповідати тим задачам і завданням , тим викликам , які сьогодні стоять перед нашою країною .


In [56]:
def spacy_ner(text):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

In [57]:
def apply_rules(text):
    ents = []

    # ORG-like (українські інституції)
    org_patterns = [
        r"Верховна Рада",
        r"Кабінет Міністрів",
        r"Міністерство [А-ЯІЇЄа-яіїє]+",
        r"Комітет [А-ЯІЇЄа-яіїє ]+",
        r"Рада [А-ЯІЇЄа-яіїє ]+"
    ]

    for p in org_patterns:
        ents += [(m.group(), "ORG") for m in re.finditer(p, text)]

    # LAW / BILL
    ents += [(m.group(), "LAW") for m in re.finditer(r"\b(закон|постанова|проєкт)\b", text)]

    # NUMBERS / IDs
    ents += [(m.group(), "CARDINAL") for m in re.finditer(r"\b\d{1,6}(-[A-Z]{1,3})?\b", text)]

    # DATE (simple)
    ents += [(m.group(), "DATE") for m in re.finditer(r"\b(20\d{2}|19\d{2})\b", text)]

    return ents

In [58]:
def merge_entities(spacy_ents, rule_ents):
    seen = set()
    final = []

    for ent, label in spacy_ents + rule_ents:
        key = (ent, label)
        if key not in seen:
            final.append((ent, label))
            seen.add(key)

    return final

In [59]:
def hybrid_ner(text):
    spacy_ents = spacy_ner(text)
    rule_ents = apply_rules(text)

    hybrid = merge_entities(spacy_ents, rule_ents)

    return spacy_ents, rule_ents, hybrid

In [62]:
results = []

for text in tqdm(texts[:200]):  # обмеження для Colab
    spacy_ents, rule_ents, hybrid = hybrid_ner(text)

    if len(hybrid) > 0:
        print("="*80)
        print(text)
        print("SPACY:", spacy_ents)
        print("RULES:", rule_ents)
        print("HYBRID:", hybrid)

    results.append({
        "text": text,
        "spacy": spacy_ents,
        "rules": rule_ents,
        "hybrid": hybrid
    })

  6%|▌         | 11/200 [00:00<00:01, 108.31it/s]

Шановні колеги , Рада національної безпеки має Секретаря Ради національної безпеки , найближчим часом будуть сформовані фактично всі підрозділи РНБО , призначені заступники Секретаря Ради національної безпеки та оборони , і я сподіваюся , що Рада національної безпеки та оборони буде відповідати тим задачам і завданням , тим викликам , які сьогодні стоять перед нашою країною .
SPACY: [('Секретаря Ради', 'PERSON'), ('найближчим часом будуть', 'PERSON'), ('Секретаря Ради', 'PERSON'), ('оборони буде', 'ORG'), ('задачам і завданням', 'PERSON'), ('тим викликам', 'PERSON')]
RULES: [('Рада національної безпеки має Секретаря Ради національної безпеки ', 'ORG'), ('Рада національної безпеки та оборони буде відповідати тим задачам і завданням ', 'ORG')]
HYBRID: [('Секретаря Ради', 'PERSON'), ('найближчим часом будуть', 'PERSON'), ('оборони буде', 'ORG'), ('задачам і завданням', 'PERSON'), ('тим викликам', 'PERSON'), ('Рада національної безпеки має Секретаря Ради національної безпеки ', 'ORG'), ('Р

 21%|██        | 42/200 [00:00<00:01, 92.82it/s]

Я би просив зараз кожного керівника силових структур чітко проінформувати про ситуацію в АРК про план заходів , як вжитих , так і тих , що необхідно негайно запроваджувати .
SPACY: [('АРК', 'ORG'), ('необхідно негайно запроваджувати', 'ORG')]
RULES: []
HYBRID: [('АРК', 'ORG'), ('необхідно негайно запроваджувати', 'ORG')]
Всі просять нас не робити поспішних кроків .
SPACY: [('Всі просять', 'PERSON')]
RULES: []
HYBRID: [('Всі просять', 'PERSON')]
Можна , звичайно , підготувати заяву Міністерства закордонних справ про невиконання російською стороною Угоди про базування Чорноморського флоту Російської Федерації та військову агресію Росії , але я думаю , що це буде значно сильніший мати ефект , якщо це буде зроблено на рівні Уряду …
SPACY: [('Міністерства', 'PERSON'), ('Чорноморського', 'GPE'), ('Російської Федерації', 'PERSON'), ('Росії', 'GPE'), ('якщо це буде', 'ORG'), ('Уряду', 'ORG')]
RULES: []
HYBRID: [('Міністерства', 'PERSON'), ('Чорноморського', 'GPE'), ('Російської Федерації', 'PE

 32%|███▏      | 63/200 [00:00<00:01, 93.74it/s]

...

Зараз розглядати питання про приєднання до ПДЧ чи до НАТО взагалі навіть неможливо , тому що це взагалі нереально .
SPACY: [('Зараз', 'PERSON')]
RULES: []
HYBRID: [('Зараз', 'PERSON')]
Третій , надзвичайно небезпечний елемент , - це те , що аеропорти й інша транспортна інфраструктура Криму береться або під контроль , або блокується переодягненими , або вже не переодягненими військовослужбовцями Чорноморського флоту Російської Федерації .
SPACY: [('Третій', 'ORG'), ('Криму', 'NORP'), ('під контроль', 'GPE'), ('Чорноморського', 'GPE'), ('Російської Федерації', 'PERSON')]
RULES: []
HYBRID: [('Третій', 'ORG'), ('Криму', 'NORP'), ('під контроль', 'GPE'), ('Чорноморського', 'GPE'), ('Російської Федерації', 'PERSON')]
Я не хотів би в такому широкому загалі говорити про ці речі , але думаю , що жодна країна , в тому числі Будапештського меморандуму , не готова буде зараз допомагати Україні .
SPACY: [('але думаю', 'PERSON'), ('Будапештського', 'ORG'), ('Україні', 'NORP')]
RULES: []
HYBRID: [('а

 90%|█████████ | 180/200 [00:01<00:00, 115.42it/s]

Ось питання .
SPACY: [('Ось питання', 'PERSON')]
RULES: []
HYBRID: [('Ось питання', 'PERSON')]
Надзвичайно потужна кампанія дезінформації і дискредитації , і нагнітання суспільних настроїв з використанням кримських і російських засобів масової інформації .
SPACY: [('кампанія дезінформації і дискредитації', 'PERSON'), ('і нагнітання', 'PERSON'), ('настроїв', 'GPE'), ('і російських', 'PERSON')]
RULES: []
HYBRID: [('кампанія дезінформації і дискредитації', 'PERSON'), ('і нагнітання', 'PERSON'), ('настроїв', 'GPE'), ('і російських', 'PERSON')]
Я просив би Вас , Міністра внутрішніх справ , Генпрокурора і Міністерство оборони у закритому режимі відпрацювати технологію затримання злочинців і доставки їх до Києва .
SPACY: [('Вас', 'PERSON'), ('Міністра', 'NORP'), ('Генпрокурора і Міністерство', 'PERSON'), ('Києва', 'PERSON')]
RULES: [('Міністерство оборони', 'ORG')]
HYBRID: [('Вас', 'PERSON'), ('Міністра', 'NORP'), ('Генпрокурора і Міністерство', 'PERSON'), ('Києва', 'PERSON'), ('Міністерство 

100%|██████████| 200/200 [00:01<00:00, 104.16it/s]

Вони повинні зрозуміти , що українська влада їм не ворог .
SPACY: [('Вони повинні зрозуміти', 'ORG')]
RULES: []
HYBRID: [('Вони повинні зрозуміти', 'ORG')]
Нам треба розвіяти міф , що це кримчани підняли повстання проти України .
SPACY: [('Нам треба розвіяти міф', 'PERSON'), ('України', 'GPE')]
RULES: []
HYBRID: [('Нам треба розвіяти міф', 'PERSON'), ('України', 'GPE')]
От це нам і треба забезпечити інформаційним поясненням , що це не є активісти якихось партійних чи громадських структур , це є російські військові , які вже навіть не приховують свою приналежність .
SPACY: [('інформаційним поясненням', 'PERSON')]
RULES: []
HYBRID: [('інформаційним поясненням', 'PERSON')]
Це дуже важливо процитувати й інформаційно забезпечити об'єктивне сприйняття українцями та всім світом цих подій .
SPACY: [('всім світом', 'PERSON')]
RULES: []
HYBRID: [('всім світом', 'PERSON')]
Андрій Віленович , два слова по роботі з кримськими елітами .
SPACY: [('Андрій Віленович', 'ORG'), ('по роботі', 'PERSON')]
R

In [63]:
errors = []

for r in results:
    sp = len(r["spacy"])
    ru = len(r["rules"])
    hy = len(r["hybrid"])

    if sp == 0 and ru > 0:
        errors.append((r["text"], "missed_by_spacy_fixed_by_rules"))

    elif sp > 0 and ru == 0:
        errors.append((r["text"], "spacy_only"))

    elif sp == 0 and ru == 0:
        errors.append((r["text"], "missed_by_all"))

print("Errors found:", len(errors))
errors[:10]

Errors found: 190


[('Рішень , які захистять Україну , які зможуть локалізувати осередки сепаратизму , рішення , які зможуть забезпечити справедливість та покарання злочинців .',
  'spacy_only'),
 ('А крім морських сил у нас немає ніяких сухопутних ?', 'missed_by_all'),
 ('Тобто всіх , які нам підпорядковані .', 'spacy_only'),
 ('Але це - номінальна чисельність .', 'spacy_only'),
 ('Боєздатна складова ?', 'missed_by_all'),
 ('Важко відповісти .', 'missed_by_all'),
 ('Більшість серед військових - місцеві контрактники .', 'spacy_only'),
 ('Для них служба - це заробіток .', 'missed_by_all'),
 ('Настрій населення в Криму ви знаєте .', 'spacy_only'),
 ('За нашою інформацією , яка надходить з різних джерел , військове керівництво Російської Федерації реально розглядає питання анексії Автономної Республіки Крим .',
  'spacy_only')]

In [64]:
for text, err_type in errors[:15]:
    print("="*80)
    print("ERROR TYPE:", err_type)
    print(text)

ERROR TYPE: spacy_only
Рішень , які захистять Україну , які зможуть локалізувати осередки сепаратизму , рішення , які зможуть забезпечити справедливість та покарання злочинців .
ERROR TYPE: missed_by_all
А крім морських сил у нас немає ніяких сухопутних ?
ERROR TYPE: spacy_only
Тобто всіх , які нам підпорядковані .
ERROR TYPE: spacy_only
Але це - номінальна чисельність .
ERROR TYPE: missed_by_all
Боєздатна складова ?
ERROR TYPE: missed_by_all
Важко відповісти .
ERROR TYPE: spacy_only
Більшість серед військових - місцеві контрактники .
ERROR TYPE: missed_by_all
Для них служба - це заробіток .
ERROR TYPE: spacy_only
Настрій населення в Криму ви знаєте .
ERROR TYPE: spacy_only
За нашою інформацією , яка надходить з різних джерел , військове керівництво Російської Федерації реально розглядає питання анексії Автономної Республіки Крим .
ERROR TYPE: spacy_only
Плюс молодь , " срочники " , які вряд чи будуть воювати .
ERROR TYPE: spacy_only
З російського боку в Криму , окрім сил Чорноморськог

In [65]:
for r in results[:20]:
    print("="*80)
    print("TEXT:", r["text"])
    print("SPACY:", r["spacy"])
    print("HYBRID:", r["hybrid"])

TEXT: Шановні колеги , Рада національної безпеки має Секретаря Ради національної безпеки , найближчим часом будуть сформовані фактично всі підрозділи РНБО , призначені заступники Секретаря Ради національної безпеки та оборони , і я сподіваюся , що Рада національної безпеки та оборони буде відповідати тим задачам і завданням , тим викликам , які сьогодні стоять перед нашою країною .
SPACY: [('Секретаря Ради', 'PERSON'), ('найближчим часом будуть', 'PERSON'), ('Секретаря Ради', 'PERSON'), ('оборони буде', 'ORG'), ('задачам і завданням', 'PERSON'), ('тим викликам', 'PERSON')]
HYBRID: [('Секретаря Ради', 'PERSON'), ('найближчим часом будуть', 'PERSON'), ('оборони буде', 'ORG'), ('задачам і завданням', 'PERSON'), ('тим викликам', 'PERSON'), ('Рада національної безпеки має Секретаря Ради національної безпеки ', 'ORG'), ('Рада національної безпеки та оборони буде відповідати тим задачам і завданням ', 'ORG')]
TEXT: Рішень , які захистять Україну , які зможуть локалізувати осередки сепаратизму